In [ ]:
from pyreaddbc import dbc2dbf
import duckdb
import sys, os, glob
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
# Conversão de DBC para DBF (Descompactar)

pasta_arquivos = r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH"

padrao_busca = os.path.join(pasta_arquivos, "*.dbc")
arquivos     = sorted(glob.glob(padrao_busca))

for i, arquivo in enumerate(arquivos):

    nome_arquivo = os.path.basename(arquivo)

    if nome_arquivo.startswith("RDCE"):
       print(f"Processando arquivo: {arquivo}", end="")

       dbc2dbf(arquivo, arquivo.replace(".dbc", ".dbf"))

       print(" - Conversão dbf - Ok")

In [ ]:
# Leitura de arquivos DBF usando DuckDB e a extensão spatial

# Seu caminho original
arquivo_dbf = (
    r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH-Internacao\RDSP2510.dbf"
)

# Inicializa a conexão
con = duckdb.connect()

# 1. Instala e carrega a extensão spatial
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

# 2. Executa a leitura do arquivo DBF usando st_read
df = \
    con.execute("""SELECT * 
                     FROM st_read(?)
                """
               ,[arquivo_dbf]).df()

# Exibe as primeiras linhas
print(df.head())



In [ ]:
# Conversão de DBF para Parquet usando DuckDB e a extensão spatial

from glob import glob
from pathlib import Path
import duckdb

# 1. Pastas de origem e destino
pasta_origem = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH")
pasta_destino = pasta_origem / "parquet"
pasta_destino.mkdir(exist_ok=True)

# 2. Inicializa o DuckDB
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# 3. Busca todos os arquivos DBF
arquivos_dbf = glob(str(pasta_origem / "*.dbf"))

arquivos_dbf_filtered = [arq for arq in arquivos_dbf if "RDSP" in arq]
print(f"Convertendo {len(arquivos_dbf_filtered)} arquivos...")

for arq in arquivos_dbf_filtered:
    file_size = os.path.getsize(arq) / (1024 * 1024)  # Tamanho em MB
    if file_size > 0:
        p_origem = Path(arq)
        # Define o nome do arquivo parquet equivalente (ex: RDPB1104.parquet)
        p_saida = pasta_destino / f"{p_origem.stem}.parquet"

        # Formata os caminhos com barras normais para evitar problemas no GDAL/Windows
        caminho_in = str(p_origem).replace("\\", "/")
        caminho_out = str(p_saida).replace("\\", "/")

        # Executa a conversão direta de DBF para Parquet
        query = \
            f"""COPY (SELECT * FROM st_read('{caminho_in}')) 
                TO '{caminho_out}' (FORMAT PARQUET, COMPRESSION 'SNAPPY');
            """
        con.execute(query)
        print(f" Convertido: {p_origem.name} -> {p_saida.name}")

print("\nConversão concluída com sucesso!")

In [ ]:
arquivos_dbf_filtered = [arq for arq in arquivos_dbf if "RDAC" in arq]
len(arquivos_dbf_filtered)

In [ ]:
df_parquet = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH-Internacao\parquet\RDSP2510.parquet")
# df_parquet.printSchema()
# df_parquet.show(5)
df_parquet.count()

df_parquet.toPandas().to_csv(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH-Internacao\parquet\RDSP2510.csv", index=False)